# FREIGHT-MNet — Cold-OD Split and Baselines

This notebook builds and evaluates a **Cold-OD split** for the FREIGHT-MNet supervised edge dataset.

The temporal split used earlier is still important, but it is favorable to historical-prior models because most 2024 FAF OD pairs have observed 2018–2022 history. The Cold-OD split is stricter: selected validation and test OD pairs are held out as **unseen FAF OD relations**, and their earlier-year labels are not used for model training or exact OD historical priors.

This notebook implements the following experiment:

- Use `2018–2022` rows from training OD pairs for training.
- Use `2023` rows from held-out validation OD pairs for validation.
- Use `2024` rows from held-out test OD pairs for testing.
- Prevent any validation/test OD pair from appearing in training rows.
- Replace exact OD historical priors with **fallback priors** based only on training origins and destinations.
- Run non-graph cold-OD baselines before GraphSAGE/HGT/D-CQHGT.

The goal is to answer a key question:

> When exact OD history is unavailable, do current demand/zone features and fallback priors still support reliable q25/q50/q75 truck travel-time quantile prediction?

## 1. Environment setup and imports

This cell imports only the packages needed for this notebook. It intentionally avoids `sklearn` because earlier runs showed NumPy ABI conflicts in some environments. LightGBM is used when available. PyTorch is optional and is only used for the optional MLP-residual cold-OD baseline.

The notebook does not run subprocess-based preflight checks. It checks the active notebook kernel directly.

In [1]:
from __future__ import annotations

import json
import math
import os
import pickle
import random
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

# Reduce optional pandas acceleration paths that can trigger binary-extension issues
# in mixed NumPy environments.
os.environ.setdefault("NUMEXPR_MAX_THREADS", "8")
os.environ.setdefault("PANDAS_USE_NUMEXPR", "0")

import numpy as np
import pandas as pd

pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception as exc:  # pragma: no cover - notebook runtime check
    lgb = None
    LGB_AVAILABLE = False
    warnings.warn(f"LightGBM is not available. LightGBM baselines will be skipped. Error: {exc}")

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception as exc:  # pragma: no cover - notebook runtime check
    torch = None
    nn = None
    F = None
    TORCH_AVAILABLE = False
    warnings.warn(f"PyTorch is not available. Optional MLP baseline will be skipped. Error: {exc}")

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as exc:  # pragma: no cover - notebook runtime check
    plt = None
    MATPLOTLIB_AVAILABLE = False
    warnings.warn(f"Matplotlib is not available. Plot generation will be skipped. Error: {exc}")

print("Python environment summary")
print("- numpy:", np.__version__)
print("- pandas:", pd.__version__)
print("- lightgbm available:", LGB_AVAILABLE, getattr(lgb, "__version__", None) if LGB_AVAILABLE else None)
print("- torch available:", TORCH_AVAILABLE, getattr(torch, "__version__", None) if TORCH_AVAILABLE else None)
if TORCH_AVAILABLE:
    print("- cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("- cuda device:", torch.cuda.get_device_name(0))

Python environment summary
- numpy: 2.4.5
- pandas: 2.3.3
- lightgbm available: True 4.6.0
- torch available: True 2.12.0+cu126
- cuda available: True
- cuda device: NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Experiment configuration

Change only this cell when switching paths, scope, or model options.

Default paths assume your project root is:

```text
E:\NetworkOptimization\pythonProject1\Data
```

The notebook writes all output artifacts to:

```text
Data/10_experiments/cold_od_split_baselines_v1_notebook/<scope>/
```

The default cold split holds out 10% of eligible 2024 OD pairs for test and 10% of eligible 2023 OD pairs for validation. Pair selection is deterministic given `seed` and stratified by the relevant year q75 whenever possible.

In [2]:
@dataclass
class ExperimentConfig:
    # Data scope and paths.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "cold_od_split_baselines_v1_notebook"

    # Reproducibility.
    seed: int = 42
    overwrite: bool = True

    # Temporal part of the split.
    train_years: Tuple[int, ...] = (2018, 2019, 2020, 2021, 2022)
    val_year: int = 2023
    test_year: int = 2024

    # Cold-OD pair holdout rates.
    val_pair_fraction: float = 0.10
    test_pair_fraction: float = 0.10
    stratification_bins: int = 10

    # Weighting.
    use_sample_weight: bool = True
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # Fallback-prior construction.
    oof_prior_folds: int = 5

    # LightGBM model options.
    run_lightgbm: bool = True
    lgb_n_estimators: int = 2000
    lgb_learning_rate: float = 0.03
    lgb_num_leaves: int = 63
    lgb_min_child_samples: int = 50
    lgb_subsample: float = 0.90
    lgb_colsample_bytree: float = 0.90
    lgb_reg_lambda: float = 1.0
    lgb_early_stopping_rounds: int = 100
    n_jobs: int = -1

    # Ensemble tuning.
    run_ensembles: bool = True
    ensemble_grid_step: float = 0.01

    # Optional MLP cold-OD residual baseline.
    run_mlp_residual: bool = True
    mlp_max_epochs: int = 300
    mlp_patience: int = 35
    mlp_batch_size: int = 4096
    mlp_hidden_dims: Tuple[int, ...] = (256, 128, 64)
    mlp_dropout: float = 0.10
    mlp_learning_rate: float = 3e-4
    mlp_weight_decay: float = 1e-4
    mlp_lambda_iqr: float = 0.10
    mlp_grad_clip: float = 5.0

    # Output formatting.
    float_output_precision: int = 6


cfg = ExperimentConfig()

# Derived paths.
scope_file = cfg.scope
supervised_path = cfg.data_root / "08_processed" / "model_ready" / f"freight_mnet_supervised_edges_2018_2024_{scope_file}.parquet"
manifest_path = cfg.data_root / "08_processed" / "model_ready" / "_metadata" / "freight_mnet_supervised_feature_manifest.json"
output_dir = cfg.data_root / "10_experiments" / cfg.run_name / cfg.scope
models_dir = output_dir / "models"
tables_dir = output_dir / "tables"
plots_dir = output_dir / "plots"

for path in [output_dir, models_dir, tables_dir, plots_dir]:
    path.mkdir(parents=True, exist_ok=True)

print("Experiment paths")
print("- supervised_path:", supervised_path)
print("- manifest_path:", manifest_path)
print("- output_dir:", output_dir)

Experiment paths
- supervised_path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
- manifest_path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
- output_dir: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf


## 3. Reproducibility helpers and constants

This cell defines global constants, label columns, quantile levels, and seed control.

The labels are annual FAF OD truck travel-time quantiles in minutes:

```text
truck_q25, truck_q50, truck_q75
```

In [3]:
LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
QUANTILE_LEVELS = np.array([0.25, 0.50, 0.75], dtype=np.float64)
SPLIT_NAMES = ["train", "val", "test"]

ORIGIN_CANDIDATES = ["faf_orig", "origin_faf", "orig_faf", "faf_origin", "origin", "orig"]
DESTINATION_CANDIDATES = ["faf_dest", "destination_faf", "dest_faf", "faf_destination", "destination", "dest"]
YEAR_CANDIDATES = ["year", "Year", "YEAR"]

LEAKAGE_PATTERNS = [
    "truck_q25", "truck_q50", "truck_q75",
    "q25", "q50", "q75",
    "input_q", "fmi", "obs_weight_sum", "n_fmi_county_pairs",
]


def set_global_seed(seed: int) -> None:
    """Set random seeds for reproducible pair splits and model initialization."""
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


set_global_seed(cfg.seed)
print("Seed set to", cfg.seed)

Seed set to 42


## 4. Path validation and schema utilities

The functions in this cell check that the required input files exist and identify the origin, destination, and year columns robustly.

FAF zone codes are normalized to three-character strings so that merges and pair IDs remain stable across integer/string formats.

In [4]:
def require_file(path: Path, description: str) -> None:
    """Raise a clear error if a required file is missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def find_first_existing_column(df: pd.DataFrame, candidates: Sequence[str], role: str) -> str:
    """Return the first candidate column present in a DataFrame."""
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"Could not find {role} column. Tried: {list(candidates)}")


def normalize_faf_code(value: Any) -> str:
    """Normalize FAF zone codes to three-character strings when possible."""
    if pd.isna(value):
        return "NA"
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    if text.isdigit():
        return text.zfill(3)
    return text


def make_od_pair_id(origin: pd.Series, destination: pd.Series) -> pd.Series:
    """Create a stable OD-pair identifier from normalized origin and destination codes."""
    return origin.astype(str) + "__" + destination.astype(str)


require_file(supervised_path, "model-ready supervised edge table")
require_file(manifest_path, "feature manifest")
print("Input files exist.")

Input files exist.


## 5. Load supervised table and feature manifest

This cell loads the model-ready supervised edge table and the feature manifest.

The notebook uses only the manifest-defined predictive features. FMI aggregation diagnostics such as `n_fmi_county_pairs` and `obs_weight_sum` can be used for evaluation segments and sample weights, but they are not allowed in the predictive feature vector.

In [5]:
def load_feature_manifest(path: Path) -> List[str]:
    """Load predictive feature columns from the feature manifest JSON."""
    with open(path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    if isinstance(manifest, dict):
        if "feature_columns" in manifest:
            cols = manifest["feature_columns"]
        elif "manifest_feature_columns" in manifest:
            cols = manifest["manifest_feature_columns"]
        else:
            raise KeyError("Manifest JSON does not contain 'feature_columns'.")
    elif isinstance(manifest, list):
        cols = manifest
    else:
        raise TypeError("Unsupported feature manifest format.")

    return [str(c) for c in cols]


def assert_no_feature_leakage(feature_columns: Sequence[str]) -> None:
    """Check that label or FMI diagnostic columns are not used as predictive features."""
    lower_cols = {col.lower(): col for col in feature_columns}
    bad = []
    for lower, original in lower_cols.items():
        if any(pattern in lower for pattern in LEAKAGE_PATTERNS):
            bad.append(original)
    if bad:
        raise ValueError(
            "Potential leakage features found in feature manifest: "
            + ", ".join(sorted(set(bad)))
        )


feature_columns = load_feature_manifest(manifest_path)
assert_no_feature_leakage(feature_columns)

raw_df = pd.read_parquet(supervised_path)
origin_col = find_first_existing_column(raw_df, ORIGIN_CANDIDATES, "origin FAF")
destination_col = find_first_existing_column(raw_df, DESTINATION_CANDIDATES, "destination FAF")
year_col = find_first_existing_column(raw_df, YEAR_CANDIDATES, "year")

missing_features = [c for c in feature_columns if c not in raw_df.columns]
missing_labels = [c for c in LABEL_COLUMNS if c not in raw_df.columns]
if missing_features:
    raise KeyError(f"Feature columns missing from supervised table: {missing_features[:20]}")
if missing_labels:
    raise KeyError(f"Label columns missing from supervised table: {missing_labels}")

# Standardize core columns and numeric types.
df = raw_df.copy()
df[origin_col] = df[origin_col].map(normalize_faf_code)
df[destination_col] = df[destination_col].map(normalize_faf_code)
df[year_col] = pd.to_numeric(df[year_col], errors="raise").astype(int)
df["od_pair"] = make_od_pair_id(df[origin_col], df[destination_col])

for col in LABEL_COLUMNS + feature_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for optional_col in ["n_fmi_county_pairs", "obs_weight_sum"]:
    if optional_col in df.columns:
        df[optional_col] = pd.to_numeric(df[optional_col], errors="coerce")

print("Loaded supervised table")
print("- shape:", df.shape)
print("- origin column:", origin_col)
print("- destination column:", destination_col)
print("- year column:", year_col)
print("- feature count:", len(feature_columns))
print("- year counts:")
print(df[year_col].value_counts().sort_index())

Loaded supervised table
- shape: (73972, 93)
- origin column: faf_orig
- destination column: faf_dest
- year column: year
- feature count: 64
- year counts:
year
2018     9936
2019    10704
2020    10761
2021    10721
2022    10651
2023    10625
2024    10574
Name: count, dtype: int64


## 6. Basic label and monotonicity checks

The supervised labels should satisfy:

```text
truck_q25 <= truck_q50 <= truck_q75
```

This cell checks the observed labels and records basic label summaries before any cold split is created.

In [6]:
def check_label_monotonicity(frame: pd.DataFrame, label_cols: Sequence[str]) -> Dict[str, Any]:
    """Return monotonicity diagnostics for q25/q50/q75 labels."""
    y = frame[list(label_cols)].to_numpy(dtype=np.float64)
    finite = np.isfinite(y).all(axis=1)
    crossing_25_50 = np.sum(y[finite, 0] > y[finite, 1])
    crossing_50_75 = np.sum(y[finite, 1] > y[finite, 2])
    return {
        "n_rows": int(len(frame)),
        "n_finite_label_rows": int(finite.sum()),
        "crossing_q25_q50": int(crossing_25_50),
        "crossing_q50_q75": int(crossing_50_75),
    }


label_diagnostics = check_label_monotonicity(df, LABEL_COLUMNS)
print(json.dumps(label_diagnostics, indent=2))

if label_diagnostics["crossing_q25_q50"] or label_diagnostics["crossing_q50_q75"]:
    raise ValueError("Observed labels contain quantile crossing. Please inspect preprocessing outputs.")

print("Label summary by year:")
print(df.groupby(year_col)[LABEL_COLUMNS].agg(["mean", "median"]).round(2))

{
  "n_rows": 73972,
  "n_finite_label_rows": 73972,
  "crossing_q25_q50": 0,
  "crossing_q50_q75": 0
}
Label summary by year:
     truck_q25          truck_q50          truck_q75         
          mean   median      mean   median      mean   median
year                                                         
2018   1464.21  1462.44   2178.03  2197.42   3462.71  3344.54
2019   1598.58  1540.95   2389.22  2425.84   3867.06  3802.99
2020   1546.97  1491.33   2316.96  2324.67   3771.33  3665.83
2021   1547.44  1498.63   2322.06  2331.85   3810.69  3720.15
2022   1498.49  1475.47   2246.07  2289.32   3651.55  3614.27
2023   1500.70  1480.00   2271.95  2322.23   3763.44  3731.48
2024   1577.52  1550.10   2384.52  2496.04   3908.88  3984.70


## 7. Build the Cold-OD pair split

This is the central split cell.

The split is done at the **OD-pair level**, not at the row level:

- Test pairs are sampled from OD pairs with a 2024 row.
- Validation pairs are sampled from remaining OD pairs with a 2023 row.
- Training pairs are all remaining OD pairs.
- Training rows use only 2018–2022 rows from training pairs.
- Validation rows use 2023 rows from validation pairs.
- Test rows use 2024 rows from test pairs.

All rows belonging to validation/test OD pairs in other years are ignored for model training. This prevents hidden OD-history leakage.

In [7]:
def build_pair_inventory(frame: pd.DataFrame) -> pd.DataFrame:
    """Build one row per OD pair with year-availability and risk summaries."""
    rows = []
    grouped = frame.groupby("od_pair", sort=False)
    train_year_set = set(cfg.train_years)

    for od_pair, group in grouped:
        years = set(group[year_col].astype(int).tolist())
        val_rows = group[group[year_col] == cfg.val_year]
        test_rows = group[group[year_col] == cfg.test_year]
        train_rows = group[group[year_col].isin(train_year_set)]
        rows.append({
            "od_pair": od_pair,
            "origin": group[origin_col].iloc[0],
            "destination": group[destination_col].iloc[0],
            "n_rows_all_years": int(len(group)),
            "n_train_year_rows": int(len(train_rows)),
            "has_train_year": bool(len(train_rows) > 0),
            "has_val_year": bool(len(val_rows) > 0),
            "has_test_year": bool(len(test_rows) > 0),
            "val_q75": float(val_rows["truck_q75"].median()) if len(val_rows) else np.nan,
            "test_q75": float(test_rows["truck_q75"].median()) if len(test_rows) else np.nan,
            "train_q75_median": float(train_rows["truck_q75"].median()) if len(train_rows) else np.nan,
        })

    return pd.DataFrame(rows).set_index("od_pair", drop=False)


def stratified_sample_pairs(
    pair_inventory: pd.DataFrame,
    candidate_mask: pd.Series,
    score_col: str,
    fraction: float,
    seed: int,
    n_bins: int,
) -> List[str]:
    """Sample OD pairs with approximate stratification by a score such as q75."""
    candidates = pair_inventory.loc[candidate_mask].copy()
    if candidates.empty:
        raise ValueError(f"No candidate pairs available for stratified sampling using {score_col}.")

    target_n = max(1, int(round(len(candidates) * fraction)))
    target_n = min(target_n, len(candidates))
    rng = np.random.default_rng(seed)

    scores = candidates[score_col].astype(float)
    if scores.notna().sum() < max(3, min(n_bins, len(candidates))):
        selected = rng.choice(candidates.index.to_numpy(), size=target_n, replace=False)
        return [str(x) for x in selected]

    filled_scores = scores.fillna(scores.median())
    try:
        # Ranking first reduces qcut failures caused by many tied q75 values.
        bins = pd.qcut(
            filled_scores.rank(method="first"),
            q=min(n_bins, len(candidates)),
            labels=False,
            duplicates="drop",
        )
    except Exception:
        selected = rng.choice(candidates.index.to_numpy(), size=target_n, replace=False)
        return [str(x) for x in selected]

    selected_pairs: List[str] = []
    for _, group in candidates.groupby(bins):
        if group.empty:
            continue
        n_group = max(1, int(round(len(group) / len(candidates) * target_n)))
        n_group = min(n_group, len(group))
        chosen = rng.choice(group.index.to_numpy(), size=n_group, replace=False)
        selected_pairs.extend([str(x) for x in chosen])

    selected_pairs = list(dict.fromkeys(selected_pairs))

    if len(selected_pairs) > target_n:
        selected_pairs = list(rng.choice(np.array(selected_pairs), size=target_n, replace=False))
    elif len(selected_pairs) < target_n:
        remaining = candidates.index.difference(selected_pairs)
        if len(remaining) > 0:
            fill_n = min(target_n - len(selected_pairs), len(remaining))
            fill = rng.choice(remaining.to_numpy(), size=fill_n, replace=False)
            selected_pairs.extend([str(x) for x in fill])

    return selected_pairs


def build_cold_od_split(frame: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, pd.Series], Dict[str, Any]]:
    """Create train/val/test masks for the cold-OD experiment."""
    pair_inventory = build_pair_inventory(frame)

    test_candidates = pair_inventory["has_test_year"]
    test_pairs = set(str(x) for x in stratified_sample_pairs(
        pair_inventory=pair_inventory,
        candidate_mask=test_candidates,
        score_col="test_q75",
        fraction=cfg.test_pair_fraction,
        seed=cfg.seed + 100,
        n_bins=cfg.stratification_bins,
    ))

    val_candidates = pair_inventory["has_val_year"] & (~pair_inventory.index.isin(test_pairs))
    val_pairs = set(str(x) for x in stratified_sample_pairs(
        pair_inventory=pair_inventory,
        candidate_mask=val_candidates,
        score_col="val_q75",
        fraction=cfg.val_pair_fraction,
        seed=cfg.seed + 200,
        n_bins=cfg.stratification_bins,
    ))

    train_pairs = set(pair_inventory.index.astype(str)) - val_pairs - test_pairs

    train_mask = frame["od_pair"].isin(train_pairs) & frame[year_col].isin(cfg.train_years)
    val_mask = frame["od_pair"].isin(val_pairs) & (frame[year_col] == cfg.val_year)
    test_mask = frame["od_pair"].isin(test_pairs) & (frame[year_col] == cfg.test_year)

    # Hard leakage guards.
    assert set(frame.loc[train_mask, "od_pair"]).isdisjoint(val_pairs), "Validation OD pair found in train rows."
    assert set(frame.loc[train_mask, "od_pair"]).isdisjoint(test_pairs), "Test OD pair found in train rows."
    assert val_pairs.isdisjoint(test_pairs), "Validation and test pair sets overlap."

    split_masks = {"train": train_mask, "val": val_mask, "test": test_mask}
    split_summary = {
        "n_pairs_total": int(len(pair_inventory)),
        "n_train_pairs": int(len(train_pairs)),
        "n_val_pairs": int(len(val_pairs)),
        "n_test_pairs": int(len(test_pairs)),
        "n_train_rows": int(train_mask.sum()),
        "n_val_rows": int(val_mask.sum()),
        "n_test_rows": int(test_mask.sum()),
        "n_ignored_rows": int((~(train_mask | val_mask | test_mask)).sum()),
        "train_years": list(cfg.train_years),
        "val_year": cfg.val_year,
        "test_year": cfg.test_year,
    }

    pair_inventory["cold_split_pair_role"] = "ignored_or_train_candidate"
    pair_inventory.loc[list(train_pairs), "cold_split_pair_role"] = "train_pair"
    pair_inventory.loc[list(val_pairs), "cold_split_pair_role"] = "val_cold_pair"
    pair_inventory.loc[list(test_pairs), "cold_split_pair_role"] = "test_cold_pair"

    return pair_inventory, split_masks, split_summary


pair_inventory, split_masks, split_summary = build_cold_od_split(df)
print(json.dumps(split_summary, indent=2))

# Attach a row-level split label for downstream saving and diagnostics.
df["cold_od_split"] = "ignored"
for split_name, mask in split_masks.items():
    df.loc[mask, "cold_od_split"] = split_name

pair_inventory.to_csv(tables_dir / "cold_od_pair_split.csv", index=False)
pd.DataFrame([split_summary]).to_csv(tables_dir / "cold_od_split_summary.csv", index=False)

print("Row split counts:")
print(df["cold_od_split"].value_counts())

{
  "n_pairs_total": 10762,
  "n_train_pairs": 8748,
  "n_val_pairs": 957,
  "n_test_pairs": 1057,
  "n_train_rows": 42849,
  "n_val_rows": 957,
  "n_test_rows": 1057,
  "n_ignored_rows": 29109,
  "train_years": [
    2018,
    2019,
    2020,
    2021,
    2022
  ],
  "val_year": 2023,
  "test_year": 2024
}
Row split counts:
cold_od_split
train      42849
ignored    29109
test        1057
val          957
Name: count, dtype: int64


## 8. Sample weights

Rows with stronger FMI support can receive larger weight in the loss. The weight is derived from `obs_weight_sum` when available:

```text
weight = log(1 + obs_weight_sum), normalized by the training mean, then clipped.
```

If the column is unavailable, all weights are set to 1. The weight column is not used as a predictive feature.

In [8]:
def compute_sample_weights(frame: pd.DataFrame, train_mask: pd.Series) -> pd.Series:
    """Compute train-normalized sample weights for all rows."""
    if (not cfg.use_sample_weight) or (cfg.weight_column not in frame.columns):
        return pd.Series(1.0, index=frame.index, name="sample_weight")

    raw = pd.to_numeric(frame[cfg.weight_column], errors="coerce").fillna(0.0).clip(lower=0.0)
    weights = np.log1p(raw)
    train_mean = float(weights.loc[train_mask].replace([np.inf, -np.inf], np.nan).dropna().mean())
    if not np.isfinite(train_mean) or train_mean <= 0:
        train_mean = 1.0

    weights = weights / train_mean
    weights = weights.clip(cfg.weight_clip_min, cfg.weight_clip_max)
    return pd.Series(weights.astype(np.float64), index=frame.index, name="sample_weight")


df["sample_weight"] = compute_sample_weights(df, split_masks["train"])
print("Sample weight summary by split:")
print(df[df["cold_od_split"].isin(SPLIT_NAMES)].groupby("cold_od_split")["sample_weight"].describe().round(3))

Sample weight summary by split:
                 count   mean    std    min    25%    50%    75%    max
cold_od_split                                                          
test            1057.0  0.961  0.325  0.125  0.735  0.955  1.185  2.004
train          42849.0  1.000  0.339  0.125  0.759  1.005  1.244  2.062
val              957.0  0.987  0.338  0.125  0.770  0.988  1.237  1.834


## 9. Cold fallback-prior construction

Exact OD historical priors are not valid for cold validation/test OD pairs, because those OD pairs must be unseen during training.

This notebook instead constructs fallback priors from training rows only:

- **Global prior:** train median q25/q50/q75.
- **Origin prior:** train median by origin FAF zone.
- **Destination prior:** train median by destination FAF zone.
- **Origin-destination blend prior:** average origin and destination priors when both exist; otherwise use the available one; otherwise use the global prior.

For training rows, origin/destination priors are generated out-of-fold by OD pair. This reduces target-encoding leakage while allowing residual models to learn from fallback priors.

In [9]:
def postprocess_quantile_array(pred: np.ndarray) -> np.ndarray:
    """Clip predictions to nonnegative values and enforce q25 <= q50 <= q75."""
    arr = np.asarray(pred, dtype=np.float64).copy()
    arr = np.where(np.isfinite(arr), arr, np.nan)
    arr = np.clip(arr, 0.0, None)
    arr = np.sort(arr, axis=1)
    return arr


def train_global_medians(frame: pd.DataFrame, train_mask: pd.Series) -> Dict[str, float]:
    """Compute global train medians for q25/q50/q75."""
    med = frame.loc[train_mask, LABEL_COLUMNS].median(numeric_only=True)
    return {col: float(med[col]) for col in LABEL_COLUMNS}


def build_zone_prior_tables(fit_frame: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    """Build origin and destination median tables from a fitting frame."""
    if fit_frame.empty:
        raise ValueError("Cannot build prior tables from an empty fitting frame.")
    global_medians = {col: float(fit_frame[col].median()) for col in LABEL_COLUMNS}
    origin_table = fit_frame.groupby(origin_col, sort=False)[LABEL_COLUMNS].median()
    dest_table = fit_frame.groupby(destination_col, sort=False)[LABEL_COLUMNS].median()
    return origin_table, dest_table, global_medians


def assign_fallback_priors(
    apply_frame: pd.DataFrame,
    origin_table: pd.DataFrame,
    dest_table: pd.DataFrame,
    global_medians: Dict[str, float],
) -> pd.DataFrame:
    """Assign origin, destination, and blended fallback priors to rows."""
    out = pd.DataFrame(index=apply_frame.index)
    out["cold_has_origin_prior"] = apply_frame[origin_col].isin(origin_table.index).astype(float)
    out["cold_has_dest_prior"] = apply_frame[destination_col].isin(dest_table.index).astype(float)

    origin_values = []
    dest_values = []
    blend_values = []

    for label in LABEL_COLUMNS:
        origin_map = origin_table[label].to_dict()
        dest_map = dest_table[label].to_dict()
        origin_pred = apply_frame[origin_col].map(origin_map).astype(float)
        dest_pred = apply_frame[destination_col].map(dest_map).astype(float)

        origin_filled = origin_pred.fillna(global_medians[label])
        dest_filled = dest_pred.fillna(global_medians[label])
        has_origin = origin_pred.notna().to_numpy(dtype=bool)
        has_dest = dest_pred.notna().to_numpy(dtype=bool)

        blend = np.full(len(apply_frame), global_medians[label], dtype=np.float64)
        both = has_origin & has_dest
        only_origin = has_origin & (~has_dest)
        only_dest = (~has_origin) & has_dest
        blend[both] = 0.5 * (origin_pred.to_numpy()[both] + dest_pred.to_numpy()[both])
        blend[only_origin] = origin_pred.to_numpy()[only_origin]
        blend[only_dest] = dest_pred.to_numpy()[only_dest]

        out[f"cold_origin_{label}"] = origin_filled.to_numpy(dtype=np.float64)
        out[f"cold_dest_{label}"] = dest_filled.to_numpy(dtype=np.float64)
        out[f"cold_prior_{label}"] = blend

        origin_values.append(out[f"cold_origin_{label}"].to_numpy())
        dest_values.append(out[f"cold_dest_{label}"].to_numpy())
        blend_values.append(out[f"cold_prior_{label}"].to_numpy())

    # Enforce quantile monotonicity for each prior block.
    origin_pp = postprocess_quantile_array(np.column_stack(origin_values))
    dest_pp = postprocess_quantile_array(np.column_stack(dest_values))
    blend_pp = postprocess_quantile_array(np.column_stack(blend_values))

    for j, label in enumerate(LABEL_COLUMNS):
        out[f"cold_origin_{label}"] = origin_pp[:, j]
        out[f"cold_dest_{label}"] = dest_pp[:, j]
        out[f"cold_prior_{label}"] = blend_pp[:, j]

    out["cold_has_any_zone_prior"] = ((out["cold_has_origin_prior"] + out["cold_has_dest_prior"]) > 0).astype(float)
    out["cold_prior_iqr"] = out["cold_prior_truck_q75"] - out["cold_prior_truck_q25"]
    out["cold_prior_q75_q50_gap"] = out["cold_prior_truck_q75"] - out["cold_prior_truck_q50"]
    out["cold_prior_q50_q25_gap"] = out["cold_prior_truck_q50"] - out["cold_prior_truck_q25"]
    return out


def make_oof_fold_assignments(train_pairs: Sequence[str], n_folds: int, seed: int) -> Dict[str, int]:
    """Assign training OD pairs to deterministic out-of-fold prior folds."""
    rng = np.random.default_rng(seed)
    pairs = np.array(sorted(set(str(p) for p in train_pairs)))
    rng.shuffle(pairs)
    n_folds = max(2, min(n_folds, len(pairs)))
    return {str(pair): int(i % n_folds) for i, pair in enumerate(pairs)}


def build_cold_fallback_prior_features(frame: pd.DataFrame, split_masks: Dict[str, pd.Series]) -> pd.DataFrame:
    """Create fallback-prior columns for all assigned split rows."""
    prior_df = pd.DataFrame(index=frame.index)
    train_mask = split_masks["train"]
    val_mask = split_masks["val"]
    test_mask = split_masks["test"]
    train_frame = frame.loc[train_mask].copy()

    full_origin_table, full_dest_table, full_global = build_zone_prior_tables(train_frame)

    # Validation and test priors are computed from all training rows only.
    for mask in [val_mask, test_mask]:
        assigned = assign_fallback_priors(frame.loc[mask], full_origin_table, full_dest_table, full_global)
        prior_df.loc[mask, assigned.columns] = assigned

    # Training priors are computed out-of-fold by OD pair to reduce target-encoding leakage.
    train_pairs = frame.loc[train_mask, "od_pair"].unique().tolist()
    fold_assignments = make_oof_fold_assignments(train_pairs, cfg.oof_prior_folds, cfg.seed + 300)
    n_folds = max(fold_assignments.values()) + 1 if fold_assignments else 1

    for fold_id in range(n_folds):
        fold_pairs = {p for p, f in fold_assignments.items() if f == fold_id}
        apply_mask = train_mask & frame["od_pair"].isin(fold_pairs)
        fit_mask = train_mask & (~frame["od_pair"].isin(fold_pairs))
        if fit_mask.sum() == 0:
            fit_mask = train_mask
        origin_table, dest_table, global_medians = build_zone_prior_tables(frame.loc[fit_mask])
        assigned = assign_fallback_priors(frame.loc[apply_mask], origin_table, dest_table, global_medians)
        prior_df.loc[apply_mask, assigned.columns] = assigned

    return prior_df


prior_features_df = build_cold_fallback_prior_features(df, split_masks)
for col in prior_features_df.columns:
    df[col] = prior_features_df[col]

COLD_PRIOR_FEATURE_COLUMNS = [
    "cold_prior_truck_q25",
    "cold_prior_truck_q50",
    "cold_prior_truck_q75",
    "cold_prior_iqr",
    "cold_prior_q75_q50_gap",
    "cold_prior_q50_q25_gap",
    "cold_has_origin_prior",
    "cold_has_dest_prior",
    "cold_has_any_zone_prior",
]

print("Cold fallback prior feature coverage by split:")
coverage_cols = ["cold_has_origin_prior", "cold_has_dest_prior", "cold_has_any_zone_prior"]
print(df[df["cold_od_split"].isin(SPLIT_NAMES)].groupby("cold_od_split")[coverage_cols].mean().round(4))

print("Cold prior summary by split:")
print(df[df["cold_od_split"].isin(SPLIT_NAMES)].groupby("cold_od_split")[["cold_prior_truck_q25", "cold_prior_truck_q50", "cold_prior_truck_q75"]].median().round(2))

Cold fallback prior feature coverage by split:
               cold_has_origin_prior  cold_has_dest_prior  \
cold_od_split                                               
test                             1.0                  1.0   
train                            1.0                  1.0   
val                              1.0                  1.0   

               cold_has_any_zone_prior  
cold_od_split                           
test                               1.0  
train                              1.0  
val                                1.0  
Cold prior summary by split:
               cold_prior_truck_q25  cold_prior_truck_q50  \
cold_od_split                                               
test                        1522.25               2296.65   
train                       1533.66               2302.72   
val                         1531.84               2302.62   

               cold_prior_truck_q75  
cold_od_split                        
test                        363

## 10. Numeric preprocessing

LightGBM can handle missing values, but this notebook uses explicit train-median imputation for reproducibility. The imputer is fitted on the cold-OD training rows only.

For the optional MLP baseline, the same train-imputed features are also z-score standardized using training means and standard deviations.

In [10]:
def fit_numeric_preprocessor(frame: pd.DataFrame, train_mask: pd.Series, columns: Sequence[str]) -> Dict[str, Dict[str, float]]:
    """Fit train-only medians, means, and standard deviations for numeric features."""
    train_values = frame.loc[train_mask, list(columns)].apply(pd.to_numeric, errors="coerce")
    train_values = train_values.replace([np.inf, -np.inf], np.nan)

    medians = train_values.median(axis=0, skipna=True).fillna(0.0)
    filled = train_values.fillna(medians)
    means = filled.mean(axis=0)
    stds = filled.std(axis=0).replace(0.0, 1.0).fillna(1.0)

    return {
        "median": {col: float(medians[col]) for col in columns},
        "mean": {col: float(means[col]) for col in columns},
        "std": {col: float(stds[col]) for col in columns},
    }


def transform_numeric_matrix(
    frame: pd.DataFrame,
    columns: Sequence[str],
    preprocessor: Dict[str, Dict[str, float]],
    standardize: bool = False,
) -> np.ndarray:
    """Transform a frame into a numeric matrix with train-fitted imputation and optional scaling."""
    values = frame.loc[:, list(columns)].apply(pd.to_numeric, errors="coerce")
    values = values.replace([np.inf, -np.inf], np.nan)
    med = pd.Series(preprocessor["median"])
    values = values.fillna(med)

    if standardize:
        mean = pd.Series(preprocessor["mean"])
        std = pd.Series(preprocessor["std"])
        values = (values - mean) / std

    return values.to_numpy(dtype=np.float32)


feature_columns_current = list(feature_columns)
feature_columns_with_cold_prior = feature_columns_current + COLD_PRIOR_FEATURE_COLUMNS

current_preprocessor = fit_numeric_preprocessor(df, split_masks["train"], feature_columns_current)
prior_preprocessor = fit_numeric_preprocessor(df, split_masks["train"], feature_columns_with_cold_prior)

print("Current feature count:", len(feature_columns_current))
print("Feature count with cold fallback priors:", len(feature_columns_with_cold_prior))

Current feature count: 64
Feature count with cold fallback priors: 73


## 11. Metric utilities

The core metrics are:

- `MAE_q25`, `MAE_q50`, `MAE_q75`
- `pinball_q25`, `pinball_q50`, `pinball_q75`, `pinball_mean`
- `weighted_pinball_mean`
- `IQR_MAE`
- stress subset metrics for top true q75 and top true IQR rows
- sparse-label metrics based on low `n_fmi_county_pairs`

All metrics are computed from the prediction parquet, so the leaderboard does not depend on any model-specific training logs.

In [11]:
def pinball_loss_array(y_true: np.ndarray, y_pred: np.ndarray, taus: np.ndarray = QUANTILE_LEVELS) -> np.ndarray:
    """Compute elementwise pinball loss for q25/q50/q75 predictions."""
    diff = y_true - y_pred
    return np.maximum(taus * diff, (taus - 1.0) * diff)


def weighted_mean(values: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Compute a finite weighted mean with safe fallback to an unweighted mean."""
    values = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(values)
    if weights is None:
        return float(np.nanmean(values[finite])) if finite.any() else np.nan
    weights = np.asarray(weights, dtype=np.float64)
    finite = finite & np.isfinite(weights) & (weights > 0)
    if not finite.any():
        return float(np.nanmean(values))
    return float(np.average(values[finite], weights=weights[finite]))


def evaluate_prediction_frame(pred_frame: pd.DataFrame, split_name: str, model_name: str, variant: str) -> Dict[str, Any]:
    """Evaluate one model on one split."""
    y_true = pred_frame[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float64)
    y_pred = pred_frame[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float64)
    weights = pred_frame["sample_weight"].to_numpy(dtype=np.float64) if "sample_weight" in pred_frame.columns else None

    abs_err = np.abs(y_pred - y_true)
    sq_err_q50 = (y_pred[:, 1] - y_true[:, 1]) ** 2
    pb = pinball_loss_array(y_true, y_pred)
    row_pinball = pb.mean(axis=1)
    true_iqr = y_true[:, 2] - y_true[:, 0]
    pred_iqr = y_pred[:, 2] - y_pred[:, 0]
    iqr_abs_err = np.abs(pred_iqr - true_iqr)

    result = {
        "split": split_name,
        "model": model_name,
        "variant": variant,
        "n": int(len(pred_frame)),
        "mae_q25": float(np.nanmean(abs_err[:, 0])),
        "mae_q50": float(np.nanmean(abs_err[:, 1])),
        "mae_q75": float(np.nanmean(abs_err[:, 2])),
        "rmse_q50": float(np.sqrt(np.nanmean(sq_err_q50))),
        "pinball_q25": float(np.nanmean(pb[:, 0])),
        "pinball_q50": float(np.nanmean(pb[:, 1])),
        "pinball_q75": float(np.nanmean(pb[:, 2])),
        "pinball_mean": float(np.nanmean(row_pinball)),
        "weighted_pinball_mean": weighted_mean(row_pinball, weights),
        "bias_q25": float(np.nanmean(y_pred[:, 0] - y_true[:, 0])),
        "bias_q50": float(np.nanmean(y_pred[:, 1] - y_true[:, 1])),
        "bias_q75": float(np.nanmean(y_pred[:, 2] - y_true[:, 2])),
        "iqr_mae": float(np.nanmean(iqr_abs_err)),
        "true_iqr_mean": float(np.nanmean(true_iqr)),
        "pred_iqr_mean": float(np.nanmean(pred_iqr)),
        "q50_inside_pred_q25_q75_rate": float(np.nanmean((y_true[:, 1] >= y_pred[:, 0]) & (y_true[:, 1] <= y_pred[:, 2]))),
    }

    # Split-local stress and sparse metrics.
    if len(pred_frame) > 0:
        q75_threshold = float(np.nanquantile(y_true[:, 2], 0.90))
        iqr_threshold = float(np.nanquantile(true_iqr, 0.90))
        top_q75 = y_true[:, 2] >= q75_threshold
        top_iqr = true_iqr >= iqr_threshold
        result["stress_top10_q75_threshold"] = q75_threshold
        result["stress_top10_n"] = int(top_q75.sum())
        result["stress_top10_mae_q75"] = float(np.nanmean(abs_err[top_q75, 2])) if top_q75.any() else np.nan
        result["top_iqr_threshold"] = iqr_threshold
        result["top_iqr_n"] = int(top_iqr.sum())
        result["top_iqr_mae_iqr"] = float(np.nanmean(iqr_abs_err[top_iqr])) if top_iqr.any() else np.nan
        result["top_iqr_mae_q75"] = float(np.nanmean(abs_err[top_iqr, 2])) if top_iqr.any() else np.nan

    if "n_fmi_county_pairs" in pred_frame.columns and pred_frame["n_fmi_county_pairs"].notna().any():
        sparse_threshold = float(np.nanquantile(pred_frame["n_fmi_county_pairs"].astype(float), 0.25))
        sparse = pred_frame["n_fmi_county_pairs"].astype(float).to_numpy() <= sparse_threshold
        result["sparse_bottom25_threshold"] = sparse_threshold
        result["sparse_bottom25_n"] = int(sparse.sum())
        result["sparse_bottom25_mae_q75"] = float(np.nanmean(abs_err[sparse, 2])) if sparse.any() else np.nan
        result["sparse_bottom25_iqr_mae"] = float(np.nanmean(iqr_abs_err[sparse])) if sparse.any() else np.nan

    return result

## 12. Prediction table construction

All model predictions are stored in one normalized long table. This avoids mixing artifacts from different experiments.

Each prediction row contains:

- source and model name
- split name
- FAF origin, FAF destination, OD pair, year
- true q25/q50/q75
- predicted q25/q50/q75
- sample weight and diagnostics columns when available

In [12]:
def get_split_frame(split_name: str) -> pd.DataFrame:
    """Return a copy of the supervised data for one cold-OD split."""
    return df.loc[split_masks[split_name]].copy()


split_frames = {split: get_split_frame(split) for split in SPLIT_NAMES}


def get_labels_for_split(split_name: str) -> np.ndarray:
    """Return label array for one split."""
    return split_frames[split_name][LABEL_COLUMNS].to_numpy(dtype=np.float64)


def get_base_prior_for_split(split_name: str, prefix: str = "cold_prior") -> np.ndarray:
    """Return q25/q50/q75 fallback-prior predictions for one split."""
    cols = [f"{prefix}_{label}" for label in LABEL_COLUMNS]
    return split_frames[split_name][cols].to_numpy(dtype=np.float64)


def create_prediction_frame(
    split_name: str,
    model_name: str,
    variant: str,
    pred: np.ndarray,
    raw_pred: Optional[np.ndarray] = None,
) -> pd.DataFrame:
    """Create a normalized prediction DataFrame for one model and split."""
    frame = split_frames[split_name].copy()
    y = frame[LABEL_COLUMNS].to_numpy(dtype=np.float64)
    pred_pp = postprocess_quantile_array(pred)
    if raw_pred is None:
        raw_pred = pred
    raw_pred = np.asarray(raw_pred, dtype=np.float64)
    raw_crossing = (raw_pred[:, 0] > raw_pred[:, 1]) | (raw_pred[:, 1] > raw_pred[:, 2])

    out = pd.DataFrame({
        "source": "ColdOD",
        "model": model_name,
        "variant": variant,
        "split": split_name,
        "faf_orig": frame[origin_col].to_numpy(),
        "faf_dest": frame[destination_col].to_numpy(),
        "od_pair": frame["od_pair"].to_numpy(),
        "year": frame[year_col].to_numpy(dtype=int),
        "true_q25": y[:, 0],
        "true_q50": y[:, 1],
        "true_q75": y[:, 2],
        "pred_q25": pred_pp[:, 0],
        "pred_q50": pred_pp[:, 1],
        "pred_q75": pred_pp[:, 2],
        "raw_crossing": raw_crossing.astype(int),
        "sample_weight": frame["sample_weight"].to_numpy(dtype=np.float64),
    })

    for optional_col in ["n_fmi_county_pairs", "obs_weight_sum"]:
        if optional_col in frame.columns:
            out[optional_col] = frame[optional_col].to_numpy()

    out["true_iqr"] = out["true_q75"] - out["true_q25"]
    out["pred_iqr"] = out["pred_q75"] - out["pred_q25"]
    out["abs_error_q25"] = np.abs(out["pred_q25"] - out["true_q25"])
    out["abs_error_q50"] = np.abs(out["pred_q50"] - out["true_q50"])
    out["abs_error_q75"] = np.abs(out["pred_q75"] - out["true_q75"])
    out["abs_error_iqr"] = np.abs(out["pred_iqr"] - out["true_iqr"])

    pb = pinball_loss_array(
        out[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float64),
        out[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float64),
    )
    out["pinball_q25"] = pb[:, 0]
    out["pinball_q50"] = pb[:, 1]
    out["pinball_q75"] = pb[:, 2]
    out["pinball_mean_row"] = pb.mean(axis=1)
    return out


all_prediction_frames: List[pd.DataFrame] = []
model_notes: Dict[str, Any] = {}

## 13. Reference prior baselines

This cell creates cold-OD reference baselines that do not use exact OD history:

1. `global_train_median`
2. `origin_prior`
3. `destination_prior`
4. `origin_destination_blend_prior`

The origin-destination blend prior is the base prior for residual LightGBM and optional residual MLP models.

In [13]:
def add_constant_or_prior_baseline(model_name: str, variant: str, split_to_pred: Dict[str, np.ndarray], note: Dict[str, Any]) -> None:
    """Add baseline predictions for all splits to the global prediction list."""
    model_notes[f"{model_name}::{variant}"] = note
    for split in SPLIT_NAMES:
        pred = split_to_pred[split]
        all_prediction_frames.append(create_prediction_frame(split, model_name, variant, pred))


global_medians = train_global_medians(df, split_masks["train"])
global_pred_by_split = {
    split: np.tile(np.array([global_medians[c] for c in LABEL_COLUMNS], dtype=np.float64), (len(split_frames[split]), 1))
    for split in SPLIT_NAMES
}
add_constant_or_prior_baseline(
    "global_train_median",
    "postprocessed",
    global_pred_by_split,
    {"description": "Global q25/q50/q75 median computed from cold-OD training rows only.", "global_medians": global_medians},
)

for prior_name, prefix in [
    ("origin_prior", "cold_origin"),
    ("destination_prior", "cold_dest"),
    ("origin_destination_blend_prior", "cold_prior"),
]:
    pred_by_split = {split: get_base_prior_for_split(split, prefix=prefix) for split in SPLIT_NAMES}
    add_constant_or_prior_baseline(
        prior_name,
        "postprocessed",
        pred_by_split,
        {"description": f"Cold-OD fallback prior using {prefix} columns fitted from training rows only."},
    )

print("Added reference prior baselines:", ["global_train_median", "origin_prior", "destination_prior", "origin_destination_blend_prior"])

Added reference prior baselines: ['global_train_median', 'origin_prior', 'destination_prior', 'origin_destination_blend_prior']


## 14. LightGBM helper functions

The LightGBM baselines are the strongest non-graph competitors in the temporal setting. In the cold-OD setting, they test whether current OD demand/zone features can generalize to unseen FAF OD relations.

This notebook trains three LightGBM families:

- `lightgbm_direct_current_features`
- `prior_feature_lgbm_direct`
- `residual_lgbm_current_features`
- `residual_lgbm_prior_features`

Residual models use:

```text
final prediction = cold fallback prior + predicted residual
```

In [14]:
def lgb_params_for_quantile(tau: float) -> Dict[str, Any]:
    """Return LightGBM parameters for a quantile regression model."""
    return {
        "objective": "quantile",
        "alpha": tau,
        "n_estimators": cfg.lgb_n_estimators,
        "learning_rate": cfg.lgb_learning_rate,
        "num_leaves": cfg.lgb_num_leaves,
        "min_child_samples": cfg.lgb_min_child_samples,
        "subsample": cfg.lgb_subsample,
        "colsample_bytree": cfg.lgb_colsample_bytree,
        "reg_lambda": cfg.lgb_reg_lambda,
        "random_state": cfg.seed,
        "n_jobs": cfg.n_jobs,
        "verbosity": -1,
    }


def fit_lgbm_quantile_family(
    model_label: str,
    feature_cols: Sequence[str],
    preprocessor: Dict[str, Dict[str, float]],
    y_train: np.ndarray,
    y_val: np.ndarray,
    standardize: bool = False,
) -> Tuple[List[Any], Dict[str, np.ndarray]]:
    """Fit three LightGBM quantile models and return predictions for all splits."""
    if not LGB_AVAILABLE:
        raise RuntimeError("LightGBM is not available in this kernel.")

    X_train = transform_numeric_matrix(split_frames["train"], feature_cols, preprocessor, standardize=standardize)
    X_val = transform_numeric_matrix(split_frames["val"], feature_cols, preprocessor, standardize=standardize)
    X_test = transform_numeric_matrix(split_frames["test"], feature_cols, preprocessor, standardize=standardize)
    X_by_split = {"train": X_train, "val": X_val, "test": X_test}
    weights_train = split_frames["train"]["sample_weight"].to_numpy(dtype=np.float64)
    weights_val = split_frames["val"]["sample_weight"].to_numpy(dtype=np.float64)

    models = []
    pred_by_split = {split: np.zeros((len(split_frames[split]), 3), dtype=np.float64) for split in SPLIT_NAMES}

    for j, tau in enumerate(QUANTILE_LEVELS):
        print(f"Training {model_label} for tau={tau:.2f}")
        model = lgb.LGBMRegressor(**lgb_params_for_quantile(float(tau)))
        callbacks = [
            lgb.early_stopping(cfg.lgb_early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ]
        model.fit(
            X_train,
            y_train[:, j],
            sample_weight=weights_train,
            eval_set=[(X_val, y_val[:, j])],
            eval_sample_weight=[weights_val],
            callbacks=callbacks,
        )
        models.append(model)
        for split in SPLIT_NAMES:
            pred_by_split[split][:, j] = model.predict(X_by_split[split])

    return models, pred_by_split


def save_pickle(obj: Any, path: Path) -> None:
    """Save a Python object with pickle."""
    with open(path, "wb") as f:
        pickle.dump(obj, f)

## 15. Train LightGBM cold-OD baselines

This cell trains direct and residual LightGBM baselines if LightGBM is available and enabled in the configuration.

If LightGBM is unavailable, the notebook still produces prior-baseline outputs.

In [15]:
if cfg.run_lightgbm and LGB_AVAILABLE:
    y_train = get_labels_for_split("train")
    y_val = get_labels_for_split("val")

    # Direct LightGBM using current manifest features only.
    direct_models, direct_pred = fit_lgbm_quantile_family(
        model_label="lightgbm_direct_current_features",
        feature_cols=feature_columns_current,
        preprocessor=current_preprocessor,
        y_train=y_train,
        y_val=y_val,
    )
    model_notes["lightgbm_direct_current_features::postprocessed"] = {
        "description": "Direct LightGBM quantile regression using current manifest features only.",
        "features": feature_columns_current,
    }
    save_pickle(direct_models, models_dir / "lightgbm_direct_current_features.pkl")
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "lightgbm_direct_current_features", "postprocessed", direct_pred[split], raw_pred=direct_pred[split]))

    # Direct LightGBM using current features plus cold fallback-prior features.
    prior_direct_models, prior_direct_pred = fit_lgbm_quantile_family(
        model_label="prior_feature_lgbm_direct",
        feature_cols=feature_columns_with_cold_prior,
        preprocessor=prior_preprocessor,
        y_train=y_train,
        y_val=y_val,
    )
    model_notes["prior_feature_lgbm_direct::postprocessed"] = {
        "description": "Direct LightGBM using current features plus cold fallback-prior features.",
        "features": feature_columns_with_cold_prior,
    }
    save_pickle(prior_direct_models, models_dir / "prior_feature_lgbm_direct.pkl")
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "prior_feature_lgbm_direct", "postprocessed", prior_direct_pred[split], raw_pred=prior_direct_pred[split]))

    # Residual LightGBM using current features only.
    base_train = get_base_prior_for_split("train", prefix="cold_prior")
    base_val = get_base_prior_for_split("val", prefix="cold_prior")
    residual_train = y_train - base_train
    residual_val = y_val - base_val
    residual_current_models, residual_current_pred_resid = fit_lgbm_quantile_family(
        model_label="residual_lgbm_current_features",
        feature_cols=feature_columns_current,
        preprocessor=current_preprocessor,
        y_train=residual_train,
        y_val=residual_val,
    )
    residual_current_pred = {
        split: get_base_prior_for_split(split, prefix="cold_prior") + residual_current_pred_resid[split]
        for split in SPLIT_NAMES
    }
    model_notes["residual_lgbm_current_features::postprocessed"] = {
        "description": "Cold fallback prior plus LightGBM residual correction using current manifest features.",
        "base_model": "origin_destination_blend_prior",
        "residual_features": feature_columns_current,
    }
    save_pickle(residual_current_models, models_dir / "residual_lgbm_current_features.pkl")
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "residual_lgbm_current_features", "postprocessed", residual_current_pred[split], raw_pred=residual_current_pred[split]))

    # Residual LightGBM using current features plus fallback-prior features.
    residual_prior_models, residual_prior_pred_resid = fit_lgbm_quantile_family(
        model_label="residual_lgbm_prior_features",
        feature_cols=feature_columns_with_cold_prior,
        preprocessor=prior_preprocessor,
        y_train=residual_train,
        y_val=residual_val,
    )
    residual_prior_pred = {
        split: get_base_prior_for_split(split, prefix="cold_prior") + residual_prior_pred_resid[split]
        for split in SPLIT_NAMES
    }
    model_notes["residual_lgbm_prior_features::postprocessed"] = {
        "description": "Cold fallback prior plus LightGBM residual correction using current features and cold fallback-prior features.",
        "base_model": "origin_destination_blend_prior",
        "residual_features": feature_columns_with_cold_prior,
    }
    save_pickle(residual_prior_models, models_dir / "residual_lgbm_prior_features.pkl")
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "residual_lgbm_prior_features", "postprocessed", residual_prior_pred[split], raw_pred=residual_prior_pred[split]))

else:
    print("Skipping LightGBM baselines because LightGBM is unavailable or cfg.run_lightgbm=False.")

Training lightgbm_direct_current_features for tau=0.25


E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training lightgbm_direct_current_features for tau=0.50


E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training lightgbm_direct_current_features for tau=0.75


...

E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training residual_lgbm_prior_features for tau=0.50


E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training residual_lgbm_prior_features for tau=0.75


E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 16. Validation-tuned ensembles

Simple convex ensembles help diagnose whether the fallback prior and direct LightGBM carry complementary information.

This cell tunes a global blend weight on validation pinball:

```text
prediction = alpha * fallback_prior + (1 - alpha) * direct_lightgbm
```

It also tunes one blend weight per quantile.

In [16]:
def compute_pinball_mean_for_arrays(y_true: np.ndarray, y_pred: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Compute mean row-wise pinball loss for arrays."""
    pb = pinball_loss_array(y_true, postprocess_quantile_array(y_pred)).mean(axis=1)
    return weighted_mean(pb, weights)


def tune_global_blend(
    base_val: np.ndarray,
    candidate_val: np.ndarray,
    y_val: np.ndarray,
    weights_val: Optional[np.ndarray],
    step: float,
) -> Tuple[float, float]:
    """Tune one alpha for all quantiles."""
    alphas = np.round(np.arange(0.0, 1.0 + step / 2.0, step), 10)
    best_alpha = 1.0
    best_score = np.inf
    for alpha in alphas:
        pred = alpha * base_val + (1.0 - alpha) * candidate_val
        score = compute_pinball_mean_for_arrays(y_val, pred, weights_val)
        if score < best_score:
            best_score = score
            best_alpha = float(alpha)
    return best_alpha, float(best_score)


def tune_per_quantile_blend(
    base_val: np.ndarray,
    candidate_val: np.ndarray,
    y_val: np.ndarray,
    weights_val: Optional[np.ndarray],
    step: float,
) -> Tuple[np.ndarray, Dict[str, float]]:
    """Tune separate alpha values for q25/q50/q75."""
    alphas = np.round(np.arange(0.0, 1.0 + step / 2.0, step), 10)
    best_alphas = np.ones(3, dtype=np.float64)
    best_scores: Dict[str, float] = {}
    for j, label in enumerate(["q25", "q50", "q75"]):
        best_score = np.inf
        best_alpha = 1.0
        for alpha in alphas:
            pred_j = alpha * base_val[:, j] + (1.0 - alpha) * candidate_val[:, j]
            diff = y_val[:, j] - pred_j
            tau = QUANTILE_LEVELS[j]
            loss = np.maximum(tau * diff, (tau - 1.0) * diff)
            score = weighted_mean(loss, weights_val)
            if score < best_score:
                best_score = score
                best_alpha = float(alpha)
        best_alphas[j] = best_alpha
        best_scores[label] = float(best_score)
    return best_alphas, best_scores


ensemble_records = []

if cfg.run_ensembles and cfg.run_lightgbm and LGB_AVAILABLE and "direct_pred" in globals():
    base_pred = {split: get_base_prior_for_split(split, prefix="cold_prior") for split in SPLIT_NAMES}
    y_val = get_labels_for_split("val")
    weights_val = split_frames["val"]["sample_weight"].to_numpy(dtype=np.float64)

    alpha_global, val_score = tune_global_blend(
        base_val=base_pred["val"],
        candidate_val=direct_pred["val"],
        y_val=y_val,
        weights_val=weights_val,
        step=cfg.ensemble_grid_step,
    )
    global_blend_pred = {
        split: alpha_global * base_pred[split] + (1.0 - alpha_global) * direct_pred[split]
        for split in SPLIT_NAMES
    }
    ensemble_records.append({
        "model": "fallback_lgbm_ensemble_global",
        "alpha_prior": alpha_global,
        "alpha_lightgbm": 1.0 - alpha_global,
        "val_weighted_pinball_mean": val_score,
    })
    model_notes["fallback_lgbm_ensemble_global::postprocessed"] = {
        "description": "Validation-tuned global convex blend of cold fallback prior and direct LightGBM.",
        "alpha_prior": alpha_global,
        "alpha_lightgbm": 1.0 - alpha_global,
    }
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "fallback_lgbm_ensemble_global", "postprocessed", global_blend_pred[split], raw_pred=global_blend_pred[split]))

    alpha_per_q, val_scores_per_q = tune_per_quantile_blend(
        base_val=base_pred["val"],
        candidate_val=direct_pred["val"],
        y_val=y_val,
        weights_val=weights_val,
        step=cfg.ensemble_grid_step,
    )
    per_q_blend_pred = {
        split: base_pred[split] * alpha_per_q.reshape(1, 3) + direct_pred[split] * (1.0 - alpha_per_q).reshape(1, 3)
        for split in SPLIT_NAMES
    }
    ensemble_records.append({
        "model": "fallback_lgbm_ensemble_per_quantile",
        "alpha_prior_q25": float(alpha_per_q[0]),
        "alpha_prior_q50": float(alpha_per_q[1]),
        "alpha_prior_q75": float(alpha_per_q[2]),
        "alpha_lightgbm_q25": float(1.0 - alpha_per_q[0]),
        "alpha_lightgbm_q50": float(1.0 - alpha_per_q[1]),
        "alpha_lightgbm_q75": float(1.0 - alpha_per_q[2]),
        **{f"val_weighted_pinball_{k}": v for k, v in val_scores_per_q.items()},
    })
    model_notes["fallback_lgbm_ensemble_per_quantile::postprocessed"] = {
        "description": "Validation-tuned per-quantile convex blend of cold fallback prior and direct LightGBM.",
        "alpha_prior_q25": float(alpha_per_q[0]),
        "alpha_prior_q50": float(alpha_per_q[1]),
        "alpha_prior_q75": float(alpha_per_q[2]),
        "alpha_lightgbm_q25": float(1.0 - alpha_per_q[0]),
        "alpha_lightgbm_q50": float(1.0 - alpha_per_q[1]),
        "alpha_lightgbm_q75": float(1.0 - alpha_per_q[2]),
    }
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "fallback_lgbm_ensemble_per_quantile", "postprocessed", per_q_blend_pred[split], raw_pred=per_q_blend_pred[split]))
else:
    print("Skipping ensembles because direct LightGBM predictions are unavailable or ensembles are disabled.")

ensemble_weights_df = pd.DataFrame(ensemble_records)
if not ensemble_weights_df.empty:
    print(ensemble_weights_df)

                                 model  alpha_prior  alpha_lightgbm  \
0        fallback_lgbm_ensemble_global          0.0             1.0   
1  fallback_lgbm_ensemble_per_quantile          NaN             NaN   

   val_weighted_pinball_mean  alpha_prior_q25  alpha_prior_q50  \
0                 195.770095              NaN              NaN   
1                        NaN              0.0              0.0   

   alpha_prior_q75  alpha_lightgbm_q25  alpha_lightgbm_q50  \
0              NaN                 NaN                 NaN   
1              0.0                 1.0                 1.0   

   alpha_lightgbm_q75  val_weighted_pinball_q25  val_weighted_pinball_q50  \
0                 NaN                       NaN                       NaN   
1                 1.0                106.377942                185.222787   

   val_weighted_pinball_q75  
0                       NaN  
1                295.795688  


## 17. Optional MLP-residual cold-OD baseline

This optional neural baseline tests whether a small MLP can learn fallback-prior residual correction in the cold-OD setting.

The model uses a residualized monotone head:

```text
q25 = softplus(base_q25 + delta_q25)
q50 = q25 + softplus(base_gap_50_25 + delta_gap_50_25)
q75 = q50 + softplus(base_gap_75_50 + delta_gap_75_50)
```

This keeps the prediction monotone by construction while learning corrections around the fallback prior.

In [17]:
if TORCH_AVAILABLE:
    class ColdResidualMLP(nn.Module):
        """MLP with a residualized monotone quantile head around a fallback prior."""

        def __init__(self, input_dim: int, hidden_dims: Sequence[int], dropout: float, residual_scale: float):
            super().__init__()
            layers: List[nn.Module] = []
            prev_dim = input_dim
            for hidden_dim in hidden_dims:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.GELU())
                layers.append(nn.Dropout(dropout))
                prev_dim = hidden_dim
            layers.append(nn.Linear(prev_dim, 3))
            self.net = nn.Sequential(*layers)
            self.residual_scale = float(residual_scale)

        def forward(self, x, base_q):
            raw = self.net(x)
            base_q25 = base_q[:, 0]
            base_gap_50_25 = torch.clamp(base_q[:, 1] - base_q[:, 0], min=1.0)
            base_gap_75_50 = torch.clamp(base_q[:, 2] - base_q[:, 1], min=1.0)

            q25 = F.softplus(base_q25 + self.residual_scale * raw[:, 0])
            gap_50_25 = F.softplus(base_gap_50_25 + self.residual_scale * raw[:, 1])
            gap_75_50 = F.softplus(base_gap_75_50 + self.residual_scale * raw[:, 2])
            q50 = q25 + gap_50_25
            q75 = q50 + gap_75_50
            return torch.stack([q25, q50, q75], dim=1)


    def torch_pinball_loss(pred, target, weights):
        """Weighted mean pinball loss for q25/q50/q75."""
        taus = torch.tensor([0.25, 0.50, 0.75], dtype=pred.dtype, device=pred.device).view(1, 3)
        diff = target - pred
        loss = torch.maximum(taus * diff, (taus - 1.0) * diff).mean(dim=1)
        return (loss * weights).sum() / torch.clamp(weights.sum(), min=1e-6)


    def torch_iqr_loss(pred, target, weights, scale: float):
        """Weighted SmoothL1 loss on normalized IQR width."""
        pred_iqr = (pred[:, 2] - pred[:, 0]) / scale
        target_iqr = (target[:, 2] - target[:, 0]) / scale
        raw = F.smooth_l1_loss(pred_iqr, target_iqr, reduction="none") * scale
        return (raw * weights).sum() / torch.clamp(weights.sum(), min=1e-6)


    def train_optional_mlp_residual() -> Optional[Dict[str, np.ndarray]]:
        """Train the optional MLP-residual baseline and return predictions by split."""
        if not cfg.run_mlp_residual:
            print("Skipping optional MLP-residual baseline because cfg.run_mlp_residual=False.")
            return None

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Training optional cold-OD MLP residual on", device)

        X_train = transform_numeric_matrix(split_frames["train"], feature_columns_with_cold_prior, prior_preprocessor, standardize=True)
        X_val = transform_numeric_matrix(split_frames["val"], feature_columns_with_cold_prior, prior_preprocessor, standardize=True)
        X_test = transform_numeric_matrix(split_frames["test"], feature_columns_with_cold_prior, prior_preprocessor, standardize=True)
        X_by_split = {"train": X_train, "val": X_val, "test": X_test}

        y_train = get_labels_for_split("train")
        y_val = get_labels_for_split("val")
        base_by_split = {split: get_base_prior_for_split(split, prefix="cold_prior") for split in SPLIT_NAMES}
        w_train = split_frames["train"]["sample_weight"].to_numpy(dtype=np.float32)
        w_val = split_frames["val"]["sample_weight"].to_numpy(dtype=np.float32)

        residual_scale = float(np.nanmedian(y_train[:, 2] - y_train[:, 0]))
        if not np.isfinite(residual_scale) or residual_scale <= 0:
            residual_scale = 1000.0

        model = ColdResidualMLP(
            input_dim=X_train.shape[1],
            hidden_dims=cfg.mlp_hidden_dims,
            dropout=cfg.mlp_dropout,
            residual_scale=residual_scale,
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.mlp_learning_rate, weight_decay=cfg.mlp_weight_decay)

        X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
        base_train_t = torch.tensor(base_by_split["train"], dtype=torch.float32, device=device)
        w_train_t = torch.tensor(w_train, dtype=torch.float32, device=device)

        X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)
        y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
        base_val_t = torch.tensor(base_by_split["val"], dtype=torch.float32, device=device)
        w_val_t = torch.tensor(w_val, dtype=torch.float32, device=device)

        n_train = X_train.shape[0]
        best_state = None
        best_val = np.inf
        best_epoch = -1
        bad_epochs = 0
        history = []

        for epoch in range(1, cfg.mlp_max_epochs + 1):
            model.train()
            permutation = torch.randperm(n_train, device=device)
            batch_losses = []
            for start in range(0, n_train, cfg.mlp_batch_size):
                idx = permutation[start:start + cfg.mlp_batch_size]
                pred = model(X_train_t[idx], base_train_t[idx])
                loss_q = torch_pinball_loss(pred, y_train_t[idx], w_train_t[idx])
                loss_iqr = torch_iqr_loss(pred, y_train_t[idx], w_train_t[idx], residual_scale)
                loss = loss_q + cfg.mlp_lambda_iqr * loss_iqr
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg.mlp_grad_clip)
                optimizer.step()
                batch_losses.append(float(loss.detach().cpu()))

            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_t, base_val_t)
                val_loss = torch_pinball_loss(val_pred, y_val_t, w_val_t)
                val_score = float(val_loss.detach().cpu())

            history.append({"epoch": epoch, "train_loss": float(np.mean(batch_losses)), "val_weighted_pinball": val_score})
            if val_score < best_val - 1e-6:
                best_val = val_score
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1

            if epoch % 10 == 0 or epoch == 1:
                print(f"epoch={epoch:03d} train_loss={np.mean(batch_losses):.4f} val_weighted_pinball={val_score:.4f}")
            if bad_epochs >= cfg.mlp_patience:
                print(f"Early stopping at epoch {epoch}. Best epoch {best_epoch}, best val weighted pinball {best_val:.4f}")
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        torch.save({"model_state_dict": model.state_dict(), "config": asdict(cfg), "history": history}, models_dir / "mlp_residual_prior_features_cold_od.pt")

        pred_by_split: Dict[str, np.ndarray] = {}
        model.eval()
        with torch.no_grad():
            for split in SPLIT_NAMES:
                X_t = torch.tensor(X_by_split[split], dtype=torch.float32, device=device)
                base_t = torch.tensor(base_by_split[split], dtype=torch.float32, device=device)
                pred = model(X_t, base_t).detach().cpu().numpy().astype(np.float64)
                pred_by_split[split] = pred

        pd.DataFrame(history).to_csv(tables_dir / "mlp_residual_training_history.csv", index=False)
        model_notes["mlp_residual_prior_features::postprocessed"] = {
            "description": "Optional MLP residual baseline using current features plus cold fallback-prior features.",
            "base_model": "origin_destination_blend_prior",
            "features": feature_columns_with_cold_prior,
            "best_epoch": best_epoch,
            "best_val_weighted_pinball": best_val,
        }
        return pred_by_split
else:
    def train_optional_mlp_residual() -> Optional[Dict[str, np.ndarray]]:
        """Skip the MLP baseline when PyTorch is unavailable."""
        print("Skipping optional MLP-residual baseline because PyTorch is unavailable.")
        return None


mlp_residual_pred = train_optional_mlp_residual()
if mlp_residual_pred is not None:
    for split in SPLIT_NAMES:
        all_prediction_frames.append(create_prediction_frame(split, "mlp_residual_prior_features", "postprocessed", mlp_residual_pred[split], raw_pred=mlp_residual_pred[split]))

Training optional cold-OD MLP residual on cuda


E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


epoch=001 train_loss=435.3800 val_weighted_pinball=396.2474
epoch=010 train_loss=233.9957 val_weighted_pinball=227.2123
epoch=020 train_loss=215.4566 val_weighted_pinball=215.3942
epoch=030 train_loss=208.5652 val_weighted_pinball=210.8144
epoch=040 train_loss=204.2277 val_weighted_pinball=206.4394
epoch=050 train_loss=200.9699 val_weighted_pinball=202.6396
epoch=060 train_loss=197.4190 val_weighted_pinball=200.6573
epoch=070 train_loss=194.1974 val_weighted_pinball=197.4166
epoch=080 train_loss=191.4063 val_weighted_pinball=194.4992
epoch=090 train_loss=189.6807 val_weighted_pinball=193.6089
epoch=100 train_loss=186.6568 val_weighted_pinball=190.4036
epoch=110 train_loss=184.7612 val_weighted_pinball=189.3380
epoch=120 train_loss=182.3206 val_weighted_pinball=189.6765
epoch=130 train_loss=180.4152 val_weighted_pinball=187.8520
epoch=140 train_loss=178.2825 val_weighted_pinball=185.8173
epoch=150 train_loss=176.5239 val_weighted_pinball=187.2061
epoch=160 train_loss=175.2859 val_weight

## 18. Build metrics, leaderboard, and diagnostics

This cell combines all prediction frames and computes metrics for every model on every split. The test leaderboard is sorted by `pinball_mean`.

In [18]:
if not all_prediction_frames:
    raise RuntimeError("No prediction frames were created. At least the reference priors should exist.")

predictions_all = pd.concat(all_prediction_frames, ignore_index=True)

metrics_records = []
for (split_name, model_name, variant), group in predictions_all.groupby(["split", "model", "variant"], sort=False):
    metrics_records.append(evaluate_prediction_frame(group, split_name, model_name, variant))

metrics_df = pd.DataFrame(metrics_records)
metrics_df = metrics_df.sort_values(["split", "pinball_mean", "mae_q75"], ascending=[True, True, True]).reset_index(drop=True)
leaderboard_test = metrics_df[metrics_df["split"] == "test"].sort_values(["pinball_mean", "mae_q75"]).reset_index(drop=True)
leaderboard_test.insert(0, "rank", np.arange(1, len(leaderboard_test) + 1))

print("Cold-OD test leaderboard")
display_cols = [
    "rank", "model", "variant", "n", "pinball_mean", "weighted_pinball_mean",
    "mae_q50", "mae_q75", "iqr_mae", "stress_top10_mae_q75", "sparse_bottom25_mae_q75",
]
print(leaderboard_test[display_cols].round(3).to_string(index=False))

Cold-OD test leaderboard
 rank                               model       variant    n  pinball_mean  weighted_pinball_mean  mae_q50  mae_q75  iqr_mae  stress_top10_mae_q75  sparse_bottom25_mae_q75
    1         mlp_residual_prior_features postprocessed 1057       205.925                190.296  387.468  716.893  584.197               822.346                  825.210
    2           prior_feature_lgbm_direct postprocessed 1057       221.630                208.747  409.201  764.075  645.038               876.356                  846.588
    3        residual_lgbm_prior_features postprocessed 1057       221.949                209.649  406.408  774.869  652.605               884.063                  856.639
    4    lightgbm_direct_current_features postprocessed 1057       222.340                206.929  411.386  772.291  667.688               936.040                  900.267
    5       fallback_lgbm_ensemble_global postprocessed 1057       222.340                206.929  411.386  772.291

## 19. Segment diagnostics for cold-OD test rows

The cold-OD leaderboard tells us which model is best overall. Segment diagnostics tell us where each model fails:

- true q75 deciles
- true IQR deciles
- low/high `n_fmi_county_pairs`
- origin-prior and destination-prior coverage

These diagnostics prepare the next graph-model experiment by identifying cold, sparse, and high-risk cases where graph generalization may help.

In [19]:
def add_test_segments(frame: pd.DataFrame) -> pd.DataFrame:
    """Add q75/IQR/sparse segment labels to test prediction rows."""
    out = frame.copy()
    test_truth = out.drop_duplicates(["od_pair", "year"])[["od_pair", "year", "true_q75", "true_iqr"]].copy()

    # qcut can fail with ties; ranking first makes bins stable.
    try:
        q75_bins = pd.qcut(test_truth["true_q75"].rank(method="first"), q=10, labels=[f"q75_decile_{i:02d}" for i in range(1, 11)])
    except Exception:
        q75_bins = pd.Series("q75_decile_unavailable", index=test_truth.index)
    try:
        iqr_bins = pd.qcut(test_truth["true_iqr"].rank(method="first"), q=10, labels=[f"iqr_decile_{i:02d}" for i in range(1, 11)])
    except Exception:
        iqr_bins = pd.Series("iqr_decile_unavailable", index=test_truth.index)

    test_truth["true_q75_decile"] = q75_bins.astype(str).to_numpy()
    test_truth["true_iqr_decile"] = iqr_bins.astype(str).to_numpy()
    out = out.merge(test_truth[["od_pair", "year", "true_q75_decile", "true_iqr_decile"]], on=["od_pair", "year"], how="left")

    if "n_fmi_county_pairs" in out.columns and out["n_fmi_county_pairs"].notna().any():
        try:
            sparse_bins = pd.qcut(out["n_fmi_county_pairs"].rank(method="first"), q=4, labels=["n_fmi_Q1_low", "n_fmi_Q2", "n_fmi_Q3", "n_fmi_Q4_high"])
            out["n_fmi_county_pairs_quartile"] = sparse_bins.astype(str)
        except Exception:
            out["n_fmi_county_pairs_quartile"] = "n_fmi_unavailable"
    else:
        out["n_fmi_county_pairs_quartile"] = "n_fmi_unavailable"

    return out


def summarize_by_segment(frame: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    """Summarize prediction errors by model and segment."""
    rows = []
    for (model, variant, segment), group in frame.groupby(["model", "variant", segment_col], sort=True):
        rows.append({
            "model": model,
            "variant": variant,
            "segment": segment,
            "n": int(len(group)),
            "pinball_mean": float(group["pinball_mean_row"].mean()),
            "mae_q50": float(group["abs_error_q50"].mean()),
            "mae_q75": float(group["abs_error_q75"].mean()),
            "iqr_mae": float(group["abs_error_iqr"].mean()),
            "bias_q75": float((group["pred_q75"] - group["true_q75"]).mean()),
        })
    return pd.DataFrame(rows)


test_predictions = predictions_all[predictions_all["split"] == "test"].copy()
test_predictions = add_test_segments(test_predictions)

segment_tables = {}
for segment_col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
    summary = summarize_by_segment(test_predictions, segment_col)
    segment_tables[segment_col] = summary
    summary.to_csv(tables_dir / f"segment_summary__{segment_col}.csv", index=False)

print("Created segment summaries:")
for key, value in segment_tables.items():
    print("-", key, value.shape)

Created segment summaries:
- true_q75_decile (110, 9)
- true_iqr_decile (110, 9)
- n_fmi_county_pairs_quartile (44, 9)


## 20. Optional diagnostic plots

Plots are useful for quick interpretation. If Matplotlib is unavailable, this cell is skipped safely.

In [20]:
def plot_leaderboard(metric: str, title: str, filename: str, top_n: Optional[int] = None) -> None:
    """Plot a horizontal leaderboard for a selected test metric."""
    if not MATPLOTLIB_AVAILABLE:
        return
    data = leaderboard_test.sort_values(metric, ascending=True).copy()
    if top_n is not None:
        data = data.head(top_n)
    labels = data["model"].astype(str) + "::" + data["variant"].astype(str)
    plt.figure(figsize=(10, max(4, 0.35 * len(data))))
    plt.barh(labels[::-1], data[metric].to_numpy()[::-1])
    plt.xlabel(metric)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(plots_dir / filename, dpi=180)
    plt.close()


def plot_metric_by_segment(segment_col: str, metric: str, models_to_plot: Sequence[str], title: str, filename: str) -> None:
    """Plot one metric by segment for selected models."""
    if not MATPLOTLIB_AVAILABLE:
        return
    summary = segment_tables[segment_col].copy()
    summary["model_variant"] = summary["model"] + "::" + summary["variant"]
    summary = summary[summary["model_variant"].isin(models_to_plot)]
    if summary.empty:
        return
    plt.figure(figsize=(14, 6))
    for model_variant, group in summary.groupby("model_variant"):
        group = group.sort_values("segment")
        plt.plot(group["segment"], group[metric], marker="o", label=model_variant)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.xlabel(segment_col)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plots_dir / filename, dpi=180)
    plt.close()


plot_leaderboard("pinball_mean", "Cold-OD Test Pinball Mean Leaderboard", "cold_od_leaderboard_pinball_mean.png")
plot_leaderboard("mae_q75", "Cold-OD Test q75 MAE Leaderboard", "cold_od_leaderboard_mae_q75.png")
plot_leaderboard("iqr_mae", "Cold-OD Test IQR MAE Leaderboard", "cold_od_leaderboard_iqr_mae.png")

selected_models = [
    "origin_destination_blend_prior::postprocessed",
    "lightgbm_direct_current_features::postprocessed",
    "prior_feature_lgbm_direct::postprocessed",
    "residual_lgbm_prior_features::postprocessed",
    "mlp_residual_prior_features::postprocessed",
]
plot_metric_by_segment("true_q75_decile", "mae_q75", selected_models, "Cold-OD q75 MAE by True q75 Decile", "cold_od_q75_mae_by_true_q75_decile.png")
plot_metric_by_segment("true_iqr_decile", "iqr_mae", selected_models, "Cold-OD IQR MAE by True IQR Decile", "cold_od_iqr_mae_by_true_iqr_decile.png")

print("Plot directory:", plots_dir)

Plot directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\plots


## 21. Save output artifacts

This cell writes all outputs to the configured experiment directory.

Important files:

```text
leaderboard_test.csv
metrics.csv
predictions_cold_od_all_splits.parquet
predictions_cold_od_val_test.parquet
tables/cold_od_pair_split.csv
tables/cold_od_split_summary.csv
tables/segment_summary__*.csv
feature_columns_used.json
model_notes.json
run_config.json
analysis_artifact_manifest.json
```

In [21]:
# Save core tables.
metrics_path = output_dir / "metrics.csv"
leaderboard_path = output_dir / "leaderboard_test.csv"
predictions_all_path = output_dir / "predictions_cold_od_all_splits.parquet"
predictions_val_test_path = output_dir / "predictions_cold_od_val_test.parquet"
feature_columns_path = output_dir / "feature_columns_used.json"
model_notes_path = output_dir / "model_notes.json"
run_config_path = output_dir / "run_config.json"
ensemble_weights_path = output_dir / "ensemble_weights.csv"
artifact_manifest_path = output_dir / "analysis_artifact_manifest.json"

metrics_df.to_csv(metrics_path, index=False)
leaderboard_test.to_csv(leaderboard_path, index=False)
predictions_all.to_parquet(predictions_all_path, index=False)
predictions_all[predictions_all["split"].isin(["val", "test"])].to_parquet(predictions_val_test_path, index=False)

if not ensemble_weights_df.empty:
    ensemble_weights_df.to_csv(ensemble_weights_path, index=False)

feature_payload = {
    "current_manifest_feature_columns": feature_columns_current,
    "cold_prior_feature_columns": COLD_PRIOR_FEATURE_COLUMNS,
    "feature_columns_with_cold_prior": feature_columns_with_cold_prior,
    "label_columns": LABEL_COLUMNS,
}
with open(feature_columns_path, "w", encoding="utf-8") as f:
    json.dump(feature_payload, f, indent=2)

with open(model_notes_path, "w", encoding="utf-8") as f:
    json.dump(model_notes, f, indent=2)

run_config = {
    "notebook": "Cold_OD_Split_and_Baselines.ipynb",
    "created_at_unix": time.time(),
    "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()},
    "paths": {
        "data_root": str(cfg.data_root),
        "supervised_path": str(supervised_path),
        "manifest_path": str(manifest_path),
        "output_dir": str(output_dir),
        "models_dir": str(models_dir),
    },
    "schema": {
        "origin_column": origin_col,
        "destination_column": destination_col,
        "year_column": year_col,
        "label_columns": LABEL_COLUMNS,
    },
    "split_summary": split_summary,
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "lightgbm_available": LGB_AVAILABLE,
        "lightgbm": getattr(lgb, "__version__", None) if LGB_AVAILABLE else None,
        "torch_available": TORCH_AVAILABLE,
        "torch": getattr(torch, "__version__", None) if TORCH_AVAILABLE else None,
    },
}
with open(run_config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

artifact_manifest = {
    "metrics": str(metrics_path),
    "leaderboard_test": str(leaderboard_path),
    "predictions_all_splits": str(predictions_all_path),
    "predictions_val_test": str(predictions_val_test_path),
    "feature_columns_used": str(feature_columns_path),
    "model_notes": str(model_notes_path),
    "run_config": str(run_config_path),
    "ensemble_weights": str(ensemble_weights_path) if ensemble_weights_path.exists() else None,
    "tables_dir": str(tables_dir),
    "plots_dir": str(plots_dir),
    "models_dir": str(models_dir),
}
with open(artifact_manifest_path, "w", encoding="utf-8") as f:
    json.dump(artifact_manifest, f, indent=2)

print("Saved Cold-OD experiment artifacts to:")
print(output_dir)
print("\nKey files:")
for key, value in artifact_manifest.items():
    print(f"- {key}: {value}")

Saved Cold-OD experiment artifacts to:
E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf

Key files:
- metrics: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\metrics.csv
- leaderboard_test: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv
- predictions_all_splits: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_all_splits.parquet
- predictions_val_test: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_val_test.parquet
- feature_columns_used: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\feature_columns_used.json
- model_notes: E:\NetworkOptimization\pythonProject1\Da

## 22. Interpretation checklist

After running the notebook, inspect these files first:

```text
leaderboard_test.csv
metrics.csv
tables/segment_summary__true_q75_decile.csv
tables/segment_summary__true_iqr_decile.csv
tables/segment_summary__n_fmi_county_pairs_quartile.csv
```

Recommended interpretation order:

1. Compare `origin_destination_blend_prior` with direct LightGBM.
2. Check whether `prior_feature_lgbm_direct` or `residual_lgbm_prior_features` improves cold-OD q75 and pinball.
3. Compare the optional `mlp_residual_prior_features` against LightGBM residual models if MLP was enabled.
4. Inspect high true-q75 and high true-IQR deciles.
5. Inspect low `n_fmi_county_pairs` quartiles.
6. Use the observed failure modes to decide the first GraphSAGE/HGT target split.

Expected scientific use:

- If tabular residual models are weak in cold-OD but strong in temporal split, graph models have a clear opportunity.
- If tabular residual models remain strong, graph models must demonstrate value on sparse, stress, topology, or decision-aware metrics.